# SparkClient: Batch Job with S3A Object Storage

This notebook demonstrates how to submit and manage a Spark batch job whose source script is stored in an S3-compatible object store using the `s3a://` protocol.

It covers:
- Configuring S3A credentials securely via Kubernetes secrets and endpoints for in-cluster object storage.
- Setting Hadoop-AWS and modular AWS SDK v2 dependencies.
- Submitting the `FileJob` using an `s3a://` URI.
- Monitoring job lifecycle and retrieving driver execution logs.
- Cleaning up the completed job.

## Parameters

Default parameters for notebook execution, parameterizable via Papermill.

In [ ]:
import os

S3_ENDPOINT = os.environ.get("S3_ENDPOINT", "http://seaweedfs.seaweedfs.svc.cluster.local:8333")
S3_REGION = os.environ.get("S3_REGION", "us-east-1")
S3_BUCKET = os.environ.get("S3_BUCKET", "test-bucket")
S3_PREFIX = os.environ.get("S3_PREFIX", "spark")
S3_SECRET_NAME = os.environ.get("S3_SECRET_NAME", "e2e-s3-credentials")

file_uri = f"s3a://{S3_BUCKET}/{S3_PREFIX}/spark_job.py"

## 1. Imports and Client Setup

Initialize the `SparkClient` with the target Kubernetes namespace.

In [ ]:
from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.spark import FileJob, SparkClient, SparkJobStatus

namespace = os.environ.get("SPARK_TEST_NAMESPACE", "default")
backend_config = KubernetesBackendConfig(namespace=namespace)
client = SparkClient(backend_config=backend_config)

print(f"SparkClient initialized for namespace: {namespace}")

## 2. Configure S3A & Spark Settings

Define S3A filesystem parameters and securely inject credentials from the Kubernetes Secret.

In [ ]:
s3_spark_conf = {
    "spark.jars.ivy": "/tmp/.ivy2",
    "spark.jars.packages": "org.apache.hadoop:hadoop-aws:3.4.1,software.amazon.awssdk:s3:2.24.6,software.amazon.awssdk:s3-transfer-manager:2.24.6",
    "spark.jars.excludes": "software.amazon.awssdk:bundle",
    "spark.hadoop.fs.s3a.endpoint": S3_ENDPOINT,
    "spark.hadoop.fs.s3a.path.style.access": "true",
    "spark.hadoop.fs.s3a.connection.timeout": "10000",
    "spark.hadoop.fs.s3a.endpoint.region": S3_REGION,
}

for role in ("driver", "executor"):
    for key in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"):
        s3_spark_conf[f"spark.kubernetes.{role}.secretKeyRef.{key}"] = f"{S3_SECRET_NAME}:{key}"

print(f"Target script URI: {file_uri}")
print(f"S3A Endpoint: {S3_ENDPOINT}")

## 3. Submit the S3A Batch Job

Submit the `FileJob` using the configured `s3a://` URI.

In [ ]:
job_name = client.submit_job(
    job=FileJob(
        file_source=file_uri,
        args=["10"],
    ),
    num_executors=1,
    resources_per_executor={
        "cpu": "1",
        "memory": "512Mi",
    },
    spark_conf=s3_spark_conf,
)

print(f"Job submitted successfully: {job_name}")

## 4. Wait for Job Completion

Poll the job status until it transitions to `SparkJobStatus.COMPLETED`.

In [ ]:
print(f"Waiting for {job_name} to complete execution...")
job = client.wait_for_job_status(
    job_name,
    status={SparkJobStatus.COMPLETED},
    timeout=600,
)

assert job is not None, f"Job {job_name} timed out or failed to return status"
assert job.status == SparkJobStatus.COMPLETED, f"Job failed with status: {job.status}"

print(f"Job {job_name} completed successfully! Driver Pod: {job.driver_pod_name}")

## 5. Retrieve Job Metadata

Fetch and verify the completed job metadata.

In [ ]:
job_info = client.get_job(job_name)
assert job_info is not None, f"Failed to retrieve job metadata for {job_name}"
print(f"Retrieved Job Metadata: Name={job_info.name}, Status={job_info.status}")

## 6. Stream Driver Logs

Read and display the driver pod execution output.

In [ ]:
print(f"Driver pod logs for {job_name}:")
print("-" * 70)
for line in client.get_job_logs(job_name):
    print(line.rstrip())
print("-" * 70)

## 7. Clean Up Resources

Delete the completed Spark job.

In [ ]:
print(f"Deleting job: {job_name}")
client.delete_job(job_name)
print("Cleanup complete.")